In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("="*70)
print("PART A - DATA MINING & PREPARATION")
print("="*70)

# LOAD DATASETS
movies = pd.read_csv('movie_metadata.csv')
reviews = pd.read_csv('IMDB_Dataset.csv')

print(f"\n Loaded {len(movies)} movies and {len(reviews)} reviews")

# ============================================================
# QUESTION 1: PROBLEM STATEMENT [10 MARKS]
# ============================================================

print("\n" + "="*70)
print("QUESTION 1: PROBLEM STATEMENT [10 MARKS]")
print("="*70)

problem_statement = """
ANALYTICAL PROBLEM: Predicting Movie Box Office Success from Production 
Attributes and Audience Reception

DECISION OBJECTIVE:
Movie studios invest millions in film production with uncertain financial 
returns. This project develops a predictive model to forecast whether a 
movie will achieve HIGH box office revenue (success) based on:

1. Pre-release production attributes:
   - Budget, genre, duration, director/cast popularity
   
2. Post-release audience sentiment:
   - Critical and user reviews from IMDB platform

By modeling these relationships, studios can:
  • Make data-informed green-light decisions
  • Optimize marketing budget allocation
  • Quantify financial risk exposure
  • Identify breakout potential early
  • Balance portfolio of blockbusters vs niche films

RELEVANT VARIABLES:

Production Factors (Structured Data):
  • budget: Production investment in USD (millions)
  • duration: Film runtime in minutes
  • genres: Film category/categories (Action, Drama, Comedy, etc.)
  • director_facebook_likes: Director popularity/influence
  • cast_total_facebook_likes: Combined actor popularity
  • actor_1/2/3_facebook_likes: Individual star power
  • imdb_score: Critical reception on 1-10 scale
  • content_rating: MPAA rating (G, PG, PG-13, R)
  • title_year: Release year (captures market trends)
  • num_voted_users: Audience reach/engagement

Audience Sentiment (Textual Data):
  • Review sentiment: Positive vs Negative user reviews (from 50,000 IMDB reviews)
  • Sentiment proportion: % positive reviews as success indicator

Target Variable (Derived):
  • Box Office Success: Binary classification
    - SUCCESS (1): Gross revenue ≥ median revenue for that release year
    - FAILURE (0): Gross revenue < median revenue for that release year
  
  Rationale: Using year-adjusted median accounts for inflation and 
  market size changes over time

ROLE OF DATA-DRIVEN MODELING:

Risk Quantification: Convert subjective judgments into statistical probability
Portfolio Management: Balance tentpole films against safer bets
Competitive Advantage: Identify market gaps and underserved audiences
Resource Optimization: Direct marketing budget to high-potential films
Financial Forecasting: Predict opening weekend and total revenue range
"""

print(problem_statement)

# ============================================================
# QUESTION 2: DATASET DESCRIPTION [10 MARKS]
# ============================================================

print("\n" + "="*70)
print("QUESTION 2: DATASET DESCRIPTION [10 MARKS]")
print("="*70)

dataset_description = f"""
DATASET STRUCTURE AND CHARACTERISTICS

PRIMARY DATASET: Movie Metadata (movie_metadata.csv)
─────────────────────────────────────────────────────

Records: {len(movies)} movies
Time Period: {int(movies['title_year'].min())} - {int(movies['title_year'].max())}
Geographic Scope: International (multiple countries and languages)

KEY VARIABLES:

Numerical Features:
  • budget: ${movies['budget'].min():,.0f} to ${movies['budget'].max():,.0f}
    Mean: ${movies['budget'].mean():,.0f} | Median: ${movies['budget'].median():,.0f}
  
  • gross (Box Office): ${movies['gross'].min():,.0f} to ${movies['gross'].max():,.0f}
    Mean: ${movies['gross'].mean():,.0f} | Median: ${movies['gross'].median():,.0f}
  
  • duration: {movies['duration'].min():.0f} to {movies['duration'].max():.0f} minutes
    Mean: {movies['duration'].mean():.1f} minutes
  
  • imdb_score: {movies['imdb_score'].min():.1f} to {movies['imdb_score'].max():.1f}
    Mean: {movies['imdb_score'].mean():.2f}
  
  • num_voted_users: Average {movies['num_voted_users'].mean():,.0f} votes per film
  
  • cast_total_facebook_likes: Total actor popularity metric
  
  • director_facebook_likes: Director popularity metric
  
  • movie_facebook_likes: Film-specific social engagement

Categorical Features:
  • genres: {movies['genres'].nunique()} unique genre combinations
  • content_rating: Distribution across G, PG, PG-13, R, NC-17, etc.
  • language: Primary filming language
  • country: Production country/origin
  • color: Film format (Color, Black & White, Unknown)

Temporal Features:
  • title_year: Release year (100 years of data from 1916-2016)

Textual/Metadata:
  • movie_title: Film title
  • director_name: Director identity
  • actor_1/2/3_name: Lead cast members
  • plot_keywords: Keywords from plot summary


SECONDARY DATASET: IMDB Reviews (IMDB_Dataset.csv)
──────────────────────────────────────────────────

Records: {len(reviews):,} reviews
Variables: 
  • review: Full review text (32 to 13,704 characters)
  • sentiment: Categorical label (Positive/Negative)

Sentiment Distribution:
  • Positive: {(reviews['sentiment']=='positive').sum():,} reviews (50%)
  • Negative: {(reviews['sentiment']=='negative').sum():,} reviews (50%)

This dataset provides rich textual data for NLP analysis and 
sentiment-based feature engineering.


DATA QUALITY SUMMARY

Missing Values (Movies Dataset):
"""

missing_pct = (movies.isnull().sum() / len(movies) * 100).sort_values(ascending=False)
for col in missing_pct[missing_pct > 0].head(10).index:
    count = movies[col].isnull().sum()
    pct = missing_pct[col]
    dataset_description += f"\n  • {col}: {count} ({pct:.1f}%)"

dataset_description += f"""

Reviews Dataset:
  • Complete: 0 missing values
  • Balanced: Equal positive/negative distribution
  • No data quality issues

INTEGRATION STRATEGY:

The datasets will be integrated as follows:

1. Movie Metadata Processing:
   - Remove films with missing budget or gross revenue
   - This ensures we have complete target variable (success/failure)

2. Review Sentiment Integration:
   - Calculate overall sentiment distribution from 50,000 reviews
   - Add aggregate sentiment metric to movies dataset
   - This creates a textual component in the structured dataset

3. Feature Creation:
   - {len(movies)} movies + review sentiment metric
   - Creates unified dataset with both production and reception factors

Expected Output:
  • ~4,000-4,500 complete movie records (after removing missing gross)
  • ~25-30 features (numerical + categorical + derived)
  • Binary target variable (success/failure)
"""

print(dataset_description)

# ============================================================
# QUESTION 3: DATA CLEANING & PREPARATION [10 MARKS]
# ============================================================

print("\n" + "="*70)
print("QUESTION 3: DATA CLEANING & PREPARATION [10 MARKS]")
print("="*70)

# STEP 1: Remove rows with missing budget OR gross
print("\n STEP 1: Handle Missing Target Variable")
print("-" * 70)
print(f"Movies before: {len(movies)}")
movies_clean = movies.dropna(subset=['budget', 'gross']).copy()
print(f"Movies after removing missing budget/gross: {len(movies_clean)}")
print(f"Rows removed: {len(movies) - len(movies_clean)}")

# STEP 2: Fill other missing values
print("\n STEP 2: Handle Missing Values in Features")
print("-" * 70)

# Categorical columns - fill with 'Unknown'
categorical_cols = ['content_rating', 'language', 'country', 'color', 'director_name']
for col in categorical_cols:
    if col in movies_clean.columns:
        movies_clean[col] = movies_clean[col].fillna('Unknown')
        print(f" {col}: Filled with 'Unknown'")

# Numerical columns - fill with median
numeric_cols = ['duration', 'imdb_score', 'director_facebook_likes', 
                'actor_1_facebook_likes', 'actor_2_facebook_likes', 
                'actor_3_facebook_likes', 'num_voted_users', 'num_critic_for_reviews',
                'num_user_for_reviews', 'facenumber_in_poster', 'aspect_ratio']

for col in numeric_cols:
    if col in movies_clean.columns and movies_clean[col].isnull().sum() > 0:
        median_val = movies_clean[col].median()
        movies_clean[col] = movies_clean[col].fillna(median_val)
        print(f" {col}: Filled missing with median {median_val:.0f}")

print(f"\nMissing values remaining: {movies_clean.isnull().sum().sum()}")

# STEP 3: Detect and remove outliers
print("\n STEP 3: Detect and Remove Outliers")
print("-" * 70)

def remove_outliers_iqr(data, column, iqr_multiplier=1.5):
    """Remove outliers using IQR method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - (iqr_multiplier * IQR)
    upper = Q3 + (iqr_multiplier * IQR)
    
    removed = data[(data[column] < lower) | (data[column] > upper)]
    kept = data[(data[column] >= lower) & (data[column] <= upper)]
    
    print(f"  • {column}: Removed {len(removed)} outliers")
    return kept

print(f"Rows before outlier removal: {len(movies_clean)}")

movies_clean = remove_outliers_iqr(movies_clean, 'budget', iqr_multiplier=1.5)
movies_clean = remove_outliers_iqr(movies_clean, 'gross', iqr_multiplier=1.5)
movies_clean = remove_outliers_iqr(movies_clean, 'duration', iqr_multiplier=1.5)

print(f"Rows after outlier removal: {len(movies_clean)}")

# STEP 4: Create target variable
print("\n STEP 4: Create Target Variable (Success/Failure)")
print("-" * 70)

# Success = revenue >= median for that year
movies_clean['revenue_threshold'] = movies_clean.groupby('title_year')['gross'].transform('median')
movies_clean['success'] = (movies_clean['gross'] >= movies_clean['revenue_threshold']).astype(int)
movies_clean = movies_clean.drop('revenue_threshold', axis=1)

success_counts = movies_clean['success'].value_counts()
print(f"Success class distribution:")
print(f"  • Successful (1): {success_counts[1]} films ({success_counts[1]/len(movies_clean)*100:.1f}%)")
print(f"  • Unsuccessful (0): {success_counts[0]} films ({success_counts[0]/len(movies_clean)*100:.1f}%)")

# STEP 5: Add review sentiment feature
print("\n STEP 5: Add Review Sentiment Feature")
print("-" * 70)

# Calculate average sentiment from reviews dataset
positive_reviews = (reviews['sentiment'] == 'positive').sum()
avg_sentiment = positive_reviews / len(reviews)
print(f"Overall sentiment across {len(reviews):,} reviews: {avg_sentiment:.2%} positive")

# Add to movies dataset
movies_clean['review_sentiment_score'] = avg_sentiment
print(f"  Added 'review_sentiment_score' feature = {avg_sentiment:.4f}")

# STEP 6: Data quality validation
print("\n STEP 6: Final Data Quality Check")
print("-" * 70)

print(f"\nFinal Dataset Shape: {movies_clean.shape}")
print(f"  • Rows: {movies_clean.shape[0]}")
print(f"  • Columns: {movies_clean.shape[1]}")
print(f"\nMissing values: {movies_clean.isnull().sum().sum()}")

print(f"\nKey Statistics:")
print(f"  • Budget: ${movies_clean['budget'].mean():,.0f} mean | ${movies_clean['budget'].median():,.0f} median")
print(f"  • Gross: ${movies_clean['gross'].mean():,.0f} mean | ${movies_clean['gross'].median():,.0f} median")
print(f"  • Duration: {movies_clean['duration'].mean():.1f} min average")
print(f"  • IMDB Score: {movies_clean['imdb_score'].mean():.2f} average")
print(f"  • Years covered: {int(movies_clean['title_year'].min())} - {int(movies_clean['title_year'].max())}")

# STEP 7: Save cleaned dataset
print("\n STEP 7: Save Cleaned Dataset")
print("-" * 70)

output_file = 'Ayebare.csv'
movies_clean.to_csv(output_file, index=False)

print(f" SUCCESS! Cleaned dataset saved as: {output_file}")
print(f"\nDataset Details:")
print(f"  • Total rows: {len(movies_clean)}")
print(f"  • Total columns: {movies_clean.shape[1]}")
print(f"  • File size: {movies_clean.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"  • Location: {output_file}")

# Display sample
print(f"\n Sample of Cleaned Data:")
print(movies_clean[['movie_title', 'budget', 'gross', 'imdb_score', 'duration', 'success']].head(10).to_string())

print("\n" + "="*70)
print(" PART A COMPLETE!")
print("="*70)
print(f"\nNext Step: Use '{output_file}' for PART B (Feature Engineering & Modeling)")

PART A - DATA MINING & PREPARATION

 Loaded 5043 movies and 50000 reviews

QUESTION 1: PROBLEM STATEMENT [10 MARKS]

ANALYTICAL PROBLEM: Predicting Movie Box Office Success from Production 
Attributes and Audience Reception

DECISION OBJECTIVE:
Movie studios invest millions in film production with uncertain financial 
returns. This project develops a predictive model to forecast whether a 
movie will achieve HIGH box office revenue (success) based on:

1. Pre-release production attributes:
   - Budget, genre, duration, director/cast popularity
   
2. Post-release audience sentiment:
   - Critical and user reviews from IMDB platform

By modeling these relationships, studios can:
  • Make data-informed green-light decisions
  • Optimize marketing budget allocation
  • Quantify financial risk exposure
  • Identify breakout potential early
  • Balance portfolio of blockbusters vs niche films

RELEVANT VARIABLES:

Production Factors (Structured Data):
  • budget: Production investment in US